# Подготовка данных к моделированию

## Цель

На этом этапе подготовить данные для построения моделей кредитного скоринга.

Основные задачи:

- исключить технические признаки;
- разделить данные на обучающую и валидационную выборки;
- обработать пропущенные значения;
- обработать обнаруженные аномальные значения;
- подготовить единый preprocessing pipeline;
- проверить корректность подготовленных данных.

Все операции, в которых вычисляются параметры преобразования данных
(например, медианы для заполнения пропусков), должны выполняться только
на обучающей выборке, чтобы избежать утечки информации из validation set.

## 2. Загрузка данных

In [16]:
# необходимые библеотеки
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

In [7]:
# загружаем данные
train = pd.read_csv("../data/raw/cs-training.csv")

print(f"Размер датасета: {train.shape}")

Размер датасета: (150000, 12)


In [2]:
train.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [3]:
# выделяем целевую переменную
X = train.drop(columns="SeriousDlqin2yrs")
y = train["SeriousDlqin2yrs"]

In [4]:
# убираем технический идентификатор
X = X.drop(columns="Unnamed: 0")

In [5]:
# проверка признаков
print("Признаки:", X.columns.tolist())
print("Target:", y.name)

Признаки: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']
Target: SeriousDlqin2yrs


In [9]:
# разбиение на выборки
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
# размеры выборок
print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("\nДоля положительного класса:")
print("Train:", y_train.mean())
print("Validation:", y_valid.mean())

Train: (120000, 10)
Validation: (30000, 10)

Доля положительного класса:
Train: 0.06684166666666666
Validation: 0.06683333333333333


In [12]:
# выдялем числовые признаки
numeric_features = X_train.columns.tolist()

numeric_features

['RevolvingUtilizationOfUnsecuredLines',
 'age',
 'NumberOfTime30-59DaysPastDueNotWorse',
 'DebtRatio',
 'MonthlyIncome',
 'NumberOfOpenCreditLinesAndLoans',
 'NumberOfTimes90DaysLate',
 'NumberRealEstateLoansOrLines',
 'NumberOfTime60-89DaysPastDueNotWorse',
 'NumberOfDependents']

In [13]:
# обработка для числовых признаков
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
# общий prprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features)
    ]
)

In [21]:
# делаем класс для обработки аномальных значений 

class ReplaceInvalidValues(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.late_columns = [
            "NumberOfTime30-59DaysPastDueNotWorse",
            "NumberOfTimes90DaysLate",
            "NumberOfTime60-89DaysPastDueNotWorse"
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["age"] = X["age"].replace(0, np.nan)
        for column in self.late_columns:
            X[column] = X[column].replace([96, 98], np.nan)

        return X

In [29]:
X_train_clean = ReplaceInvalidValues().fit_transform(X_train)

display(
    X_train_clean.isna().sum()
    .sort_values(ascending=False)
    .to_frame("Количество пропусков")
)

,Количество пропусков
MonthlyIncome,23675
NumberOfDependents,3128
NumberOfTime60-89DaysPastDueNotWorse,214
NumberOfTime30-59DaysPastDueNotWorse,214
NumberOfTimes90DaysLate,214
age,1
RevolvingUtilizationOfUnsecuredLines,0
DebtRatio,0
NumberOfOpenCreditLinesAndLoans,0
NumberRealEstateLoansOrLines,0


In [23]:
# собираем полный preprocessing
preprocessor = Pipeline([
    ("invalid_values", ReplaceInvalidValues()),
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]))
])

In [24]:
# проверка 
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [30]:
preprocessor.fit_transform(X_train)
preprocessor.transform(X_valid)

print("Train:", X_train_processed.shape)
print("Validation:", X_valid_processed.shape)

print("Пропуски после preprocessing:")
print("Train:", np.isnan(X_train_processed).sum())
print("Validation:", np.isnan(X_valid_processed).sum())

Train: (120000, 10)
Validation: (30000, 10)
Пропуски после preprocessing:
Train: 0
Validation: 0


## Итоги

В рамках подготовки данных:

- исключён технический идентификатор `Unnamed: 0`;
- данные разделены на обучающую и валидационную выборки в соотношении 80/20;
- при разделении сохранено исходное соотношение классов;
- значения `age = 0` и `96/98` в признаках просрочек преобразованы в пропуски;
- пропуски заполнены медианными значениями;
- числовые признаки стандартизированы;
- preprocessing обучается только на тренировочной выборке и применяется к validation без повторного обучения.